In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import re
import time
from threading import Thread


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import re
import time
from threading import Thread


In [6]:
def load_csv(file):
    """
    Reads the uploaded CSV into a pandas DataFrame.
    Returns it for display and stores in state.
    """
    path = file.name if hasattr(file, "name") else file
    df = pd.read_csv(path)
    return df, df, gr.Slider(minimum=0, maximum=len(df)-1, step=1, value=0, label="Row Index")

def plot_scatter(df):
    """
    Creates two scatter‐plots of all location_i_x/Y pairs:
     - first 1000 rows
     - last 1000 rows
    """
    # Number of rows
    n = len(df)
    first_slice = df.iloc[:1000]
    last_slice  = df.iloc[-1000:] if n >= 1000 else df

    # Extract all i indices from columns like 'location_{i}_x'
    prefixes = sorted({
        m for col in df.columns
        for m in re.findall(r'location_(\\d+)_x', col)
    }, key=int)

    # Plot #1
    fig1, ax1 = plt.subplots()
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax1.scatter(first_slice[xcol], first_slice[ycol], label=i, s=1)
    ax1.set_title("Rows 0–999")
    ax1.set_xlabel("X")
    ax1.set_ylabel("Y")
    ax1.legend(title="index")

    # Plot #2
    fig2, ax2 = plt.subplots()
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax2.scatter(last_slice[xcol], last_slice[ycol], label=i, s=1)
    ax2.set_title(f"Last {len(last_slice)} rows")
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")
    ax2.legend(title="index")

    return fig1, fig2

def plot_animation_frame(df, row_index, is_playing=False):
    """
    Creates a scatter plot showing locations for a single row of data
    with trails of previous 10 positions
    """
    if df is None or len(df) == 0:
        return None
    
    # Ensure row_index is within bounds
    row_index = min(max(0, row_index), len(df) - 1)
    
    # Extract all i indices from columns like 'location_{i}_x'
    prefixes = sorted({
        m for col in df.columns
        for m in re.findall(r'location_(\\d+)_x', col)
    }, key=int)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Get trail rows - previous 10 positions
    trail_length = 10
    trail_start = max(0, row_index - trail_length)
    trail_rows = df.iloc[trail_start:row_index+1]
    
    # Plot trails for each location
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        
        # Plot trail with fading opacity
        for t, trail_idx in enumerate(range(trail_start, row_index)):
            if trail_idx >= 0:
                # Calculate opacity based on position in trail (more recent = more opaque)
                alpha = 0.2 + 0.8 * (t / trail_length) if trail_length > 0 else 0.2
                size = 10 + 40 * (t / trail_length) if trail_length > 0 else 10
                
                # Plot trail point
                ax.scatter(
                    df.iloc[trail_idx][xcol], 
                    df.iloc[trail_idx][ycol],
                    color=f'C{int(i)}',  # Keep color consistent with location
                    alpha=alpha,
                    s=size,
                    zorder=t  # Older points are under newer ones
                )
        
        # Plot current position (full opacity)
        ax.scatter(
            df.iloc[row_index][xcol], 
            df.iloc[row_index][ycol], 
            label=f"Location {i}", 
            s=50,
            edgecolor='black',
            zorder=trail_length+1
        )
        
        # Add connecting lines for trails
        if trail_start < row_index:
            trail_x = trail_rows[xcol].values
            trail_y = trail_rows[ycol].values
            ax.plot(trail_x, trail_y, color=f'C{int(i)}', alpha=0.5, linestyle='--', linewidth=1)
    
    # Add point labels for current position
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax.annotate(f"{i}", (df.iloc[row_index][xcol], df.iloc[row_index][ycol]), 
                   xytext=(5, 5), textcoords='offset points')
    
    ax.set_title(f"Locations at Row {row_index}" + (" (Playing Animation)" if is_playing else ""))
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.legend()
    
    # Equal aspect ratio for proper spatial representation
    ax.set_aspect('equal')
    
    # If this is already part of animation and auto-range is needed:
    if len(trail_rows) > 0:
        # Find min/max coordinates to set appropriate bounds (with some padding)
        all_x_cols = [f"location_{i}_x" for i in prefixes]
        all_y_cols = [f"location_{i}_y" for i in prefixes]
        
        x_min = trail_rows[all_x_cols].min().min()
        x_max = trail_rows[all_x_cols].max().max()
        y_min = trail_rows[all_y_cols].min().min()
        y_max = trail_rows[all_y_cols].max().max()
        
        # Add some padding (10% of range)
        x_pad = 0.1 * (x_max - x_min) if x_max > x_min else 1
        y_pad = 0.1 * (y_max - y_min) if y_max > y_min else 1
        
        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
    
    return fig

# Animation state variables
animation_thread = None
stop_animation = False

def play_animation(df, row_slider, animation_plot, speed=0.1):
    global animation_thread, stop_animation
    
    # Reset stop flag
    stop_animation = False
    
    if animation_thread is not None and animation_thread.is_alive():
        # Animation already running, stop it
        stop_animation = True
        animation_thread.join()
        return "Play", row_slider, animation_plot
    
    def animation_worker():
        global stop_animation
        max_idx = len(df) - 1
        current_idx = row_slider
        
        while not stop_animation and current_idx <= max_idx:
            # Update plot with is_playing flag
            fig = plot_animation_frame(df, current_idx, True)
            
            # Update UI with new plot and slider position
            animation_plot.update(fig)
            row_slider.update(current_idx)
            
            # Sleep briefly for animation effect - use speed parameter
            time.sleep(speed)
            
            # Move to next frame
            current_idx += 1
            if current_idx > max_idx:
                # Reset to beginning for loop playback
                current_idx = 0
        
        # Reset back to "Play" when animation completes
        # This doesn't directly update the button because it's in a thread
    
    # Start animation thread
    animation_thread = Thread(target=animation_worker)
    animation_thread.start()
    
    # Return "Stop" to update button label
    return "Stop", row_slider, animation_plot

def stop_animation_fn():
    global animation_thread, stop_animation
    
    # Set flag to stop animation
    stop_animation = True
    
    # Wait for thread to finish if it's running
    if animation_thread is not None and animation_thread.is_alive():
        animation_thread.join()
    
    # Return "Play" to update button label
    return "Play"


In [7]:
with gr.Blocks() as demo:
    gr.Markdown("## CSV Scatter‐Plot: `location_i_x` vs `location_i_y`")
    
    with gr.Row():
        file_input = gr.File(label="Upload CSV", file_types=['.csv'])
        load_btn   = gr.Button("Load CSV")
    
    df_table = gr.Dataframe(label="DataFrame Preview")
    df_state = gr.State()
    
    with gr.Row():
        scatter_btn   = gr.Button("Plot scatter")
        scatter1_plot = gr.Plot(label="First 1000 rows")
        scatter2_plot = gr.Plot(label="Last 1000 rows")
    
    gr.Markdown("## Animation with 10-Row Trail")
    with gr.Row():
        row_slider = gr.Slider(minimum=0, maximum=0, step=1, value=0, label="Row Index")
        animation_plot = gr.Plot(label="Locations Animation")
    
    with gr.Row():
        play_btn = gr.Button("Play")
        speed_slider = gr.Slider(minimum=0.01, maximum=1, value=0.1, step=0.01, label="Animation Speed (sec/frame)")
    
    # Load CSV data and update components
    load_outputs = load_btn.click(
        fn=load_csv,
        inputs=file_input,
        outputs=[df_table, df_state, row_slider]
    )
    
    # Generate static scatter plots
    scatter_btn.click(
        fn=plot_scatter,
        inputs=df_state,
        outputs=[scatter1_plot, scatter2_plot]
    )
    
    # Update animation plot when slider changes
    row_slider.change(
        fn=plot_animation_frame,
        inputs=[df_state, row_slider],
        outputs=animation_plot
    )
    
    # Play/Stop button functionality
    play_btn.click(
        fn=play_animation,
        inputs=[df_state, row_slider, animation_plot],
        outputs=[play_btn, row_slider, animation_plot]
    )
    
    gr.Markdown("""
    **Tips**  
    - If you ever change the number of locations, this will auto-detect all `location_{i}_x` columns.
    - The animation slider shows the location positions for each individual row in the dataset.
    - Each location shows a trail of its 10 previous positions.
    - Use the Play button to animate through the data automatically.
    - Adjust the animation speed slider to control how fast the animation plays.
    """)

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import re

In [9]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import re

def load_csv(file):
    """
    Reads the uploaded CSV into a pandas DataFrame.
    Returns it for display and stores in state.
    """
    path = file.name if hasattr(file, "name") else file
    df = pd.read_csv(path)
    return df, df, gr.Slider(minimum=0, maximum=len(df)-1, step=1, value=0, label="Row Index")

def plot_scatter(df):
    """
    Creates two scatter‐plots of all location_i_x/Y pairs:
     - first 1000 rows
     - last 1000 rows
    """
    # Number of rows
    n = len(df)
    first_slice = df.iloc[:1000]
    last_slice  = df.iloc[-1000:] if n >= 1000 else df

    # Extract all i indices from columns like 'location_{i}_x'
    prefixes = sorted({
        m for col in df.columns
        for m in re.findall(r'location_(\d+)_x', col)
    }, key=int)

    # Plot #1
    fig1, ax1 = plt.subplots()
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax1.scatter(first_slice[xcol], first_slice[ycol], label=i, s=1)
    ax1.set_title("Rows 0–999")
    ax1.set_xlabel("X")
    ax1.set_ylabel("Y")
    ax1.legend(title="index")

    # Plot #2
    fig2, ax2 = plt.subplots()
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax2.scatter(last_slice[xcol], last_slice[ycol], label=i, s=1)
    ax2.set_title(f"Last {len(last_slice)} rows")
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")
    ax2.legend(title="index")

    return fig1, fig2

def plot_animation_frame(df, row_index):
    """
    Creates a scatter plot showing locations for a single row of data
    """
    if df is None or len(df) == 0:
        return None
    
    # Ensure row_index is within bounds
    row_index = min(max(0, row_index), len(df) - 1)
    
    # Get the specific row
    row = df.iloc[row_index]
    
    # Extract all i indices from columns like 'location_{i}_x'
    prefixes = sorted({
        m for col in df.columns
        for m in re.findall(r'location_(\d+)_x', col)
    }, key=int)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot each location point
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax.scatter(row[xcol], row[ycol], label=f"Location {i}", s=50)
        
    # Add point labels
    for i in prefixes:
        xcol = f"location_{i}_x"
        ycol = f"location_{i}_y"
        ax.annotate(f"{i}", (row[xcol], row[ycol]), 
                    xytext=(5, 5), textcoords='offset points')
    
    ax.set_title(f"Locations at Row {row_index}")
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.legend()
    
    # Equal aspect ratio for proper spatial representation
    ax.set_aspect('equal')
    
    return fig

with gr.Blocks() as demo:
    gr.Markdown("## CSV Scatter‐Plot: `location_i_x` vs `location_i_y`")
    
    with gr.Row():
        file_input = gr.File(label="Upload CSV", file_types=['.csv'])
        load_btn   = gr.Button("Load CSV")
    
    df_table = gr.Dataframe(label="DataFrame Preview")
    df_state = gr.State()
    
    with gr.Row():
        scatter_btn   = gr.Button("Plot scatter")
        scatter1_plot = gr.Plot(label="First 1000 rows")
        scatter2_plot = gr.Plot(label="Last 1000 rows")
    
    gr.Markdown("## Animation by Row")
    with gr.Row():
        row_slider = gr.Slider(minimum=0, maximum=0, step=1, value=0, label="Row Index")
        animation_plot = gr.Plot(label="Locations Animation")
    
    # Load CSV data and update components
    load_outputs = load_btn.click(
        fn=load_csv,
        inputs=file_input,
        outputs=[df_table, df_state, row_slider]
    )
    
    # Generate static scatter plots
    scatter_btn.click(
        fn=plot_scatter,
        inputs=df_state,
        outputs=[scatter1_plot, scatter2_plot]
    )
    
    # Update animation plot when slider changes
    row_slider.change(
        fn=plot_animation_frame,
        inputs=[df_state, row_slider],
        outputs=animation_plot
    )
    
    gr.Markdown("""
    **Tips**  
    - If you ever change the number of locations, this will auto-detect all `location_{i}_x` columns.  
    - You can swap the fixed 1000-row splits for user-selectable ranges with `gr.Slider` inputs.  
    - The animation slider shows the location positions for each individual row in the dataset.
    - Move the slider to see how positions change over time/rows.
    """)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
